In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import Dict, Tuple

from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.gridspec as gridspec

from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    brier_score_loss
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier


# ============================================================
# GLOBAL PLOT STYLE
# ============================================================

plt.rcParams.update({
    "figure.figsize": (7.5, 5.5),
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 16,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "--",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "savefig.dpi": 300,
    "savefig.bbox": "tight"
})


# ============================================================
# FILE PATHS
# ============================================================

HEART_PATH = "Datasets/heart/processed.cleveland.data"
BREAST_PATH = "Datasets/breast/wdbc.data"

# Example Windows paths:
# HEART_PATH = r"C:\Users\Adam\Desktop\Datasets\processed.cleveland.data"
# BREAST_PATH = r"C:\Users\Adam\Desktop\Datasets\wdbc.data"

OUTPUT_DIR = "trustworthy_ai_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
NOISE_STD = 0.10
N_SPLITS = 5
N_REPEATS = 5
N_BOOTSTRAP = 300
TOP_N_FEATURES = 10
MIN_GROUP_SIZE = 10
CLASSIFICATION_THRESHOLD = 0.5


# ============================================================
# MODEL CONFIG
# ============================================================

@dataclass
class ModelSpec:
    name: str
    estimator: object
    interpretability_prior: float


MODEL_SPECS = [
    ModelSpec(
        name="Logistic Regression",
        estimator=LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
        interpretability_prior=1.00
    ),
    ModelSpec(
        name="Random Forest",
        estimator=RandomForestClassifier(
            n_estimators=300,
            min_samples_split=4,
            random_state=RANDOM_STATE
        ),
        interpretability_prior=0.70
    ),
    ModelSpec(
        name="Gradient Boosting",
        estimator=GradientBoostingClassifier(random_state=RANDOM_STATE),
        interpretability_prior=0.60
    ),
]


# ============================================================
# DATA LOADING
# ============================================================

def load_heart_dataset(path: str) -> Tuple[pd.DataFrame, pd.Series, pd.Series]:
    heart_columns = [
        "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
        "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"
    ]

    df = pd.read_csv(path, names=heart_columns, na_values="?")
    df["target"] = (df["target"] > 0).astype(int)

    X = df.drop("target", axis=1)
    y = df["target"]

    # Subgroup for fairness analysis
    subgroup = X["sex"].map({0.0: "female", 1.0: "male"}).fillna("unknown")

    return X, y, subgroup


def load_breast_dataset(path: str) -> Tuple[pd.DataFrame, pd.Series, pd.Series]:
    breast_columns = [
        "id", "diagnosis",
        "radius_mean", "texture_mean", "perimeter_mean", "area_mean", "smoothness_mean",
        "compactness_mean", "concavity_mean", "concave_points_mean", "symmetry_mean", "fractal_dimension_mean",
        "radius_se", "texture_se", "perimeter_se", "area_se", "smoothness_se",
        "compactness_se", "concavity_se", "concave_points_se", "symmetry_se", "fractal_dimension_se",
        "radius_worst", "texture_worst", "perimeter_worst", "area_worst", "smoothness_worst",
        "compactness_worst", "concavity_worst", "concave_points_worst", "symmetry_worst", "fractal_dimension_worst"
    ]

    df = pd.read_csv(path, names=breast_columns)
    df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})

    X = df.drop(["id", "diagnosis"], axis=1)
    y = df["diagnosis"]

    # Proxy subgroup for fairness analysis
    subgroup = np.where(
        X["radius_mean"] >= X["radius_mean"].median(),
        "high_radius",
        "low_radius"
    )
    subgroup = pd.Series(subgroup, index=X.index)

    return X, y, subgroup


In [2]:
# ============================================================
# PREPROCESSING
# ============================================================

def make_pipeline(model) -> Pipeline:
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model)
    ])




# ============================================================
# Weights
# ============================================================
import numpy as np

def compute_ahp_weights():

    matrix = np.array([
        [1,   3,   3,   5,   5],
        [1/3, 1,   2,   3,   3],
        [1/3, 1/2, 1,   3,   3],
        [1/5, 1/3, 1/3, 1,   2],
        [1/5, 1/3, 1/3, 1/2, 1]
    ])

    eigvals, eigvecs = np.linalg.eig(matrix)

    max_index = np.argmax(eigvals.real)
    weights = eigvecs[:, max_index].real

    weights = weights / weights.sum()

    labels = [
        "performance",
        "robustness",
        "fairness",
        "calibration",
        "interpretability"
    ]

    return dict(zip(labels, weights))

weights = compute_ahp_weights()

print("AHP-derived weights:")
for k, v in weights.items():
    print(f"{k}: {v:.3f}")


# ============================================================
# HELPERS
# ============================================================

def sanitize_name(text: str) -> str:
    return text.lower().replace(" ", "_").replace("-", "_")


def get_probabilities(model: Pipeline, X: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]

    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
        return scores

    return model.predict(X).astype(float)


def compute_performance(
    y_true: pd.Series,
    y_prob: np.ndarray,
    threshold: float = CLASSIFICATION_THRESHOLD
) -> Dict[str, float]:
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob)
    }


def bootstrap_auc_ci(
    y_true: pd.Series,
    y_prob: np.ndarray,
    n_bootstrap: int = N_BOOTSTRAP,
    random_state: int = RANDOM_STATE
) -> Tuple[float, float]:
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    aucs = []
    n = len(y_true)

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        y_b = y_true[idx]
        p_b = y_prob[idx]

        if len(np.unique(y_b)) < 2:
            continue

        aucs.append(roc_auc_score(y_b, p_b))

    if len(aucs) == 0:
        return np.nan, np.nan

    return float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))


def compute_calibration_error(
    y_true: pd.Series,
    y_prob: np.ndarray,
    n_bins: int = 10
) -> float:
    frac_pos, mean_pred = calibration_curve(
        y_true,
        y_prob,
        n_bins=n_bins,
        strategy="uniform"
    )

    if len(frac_pos) == 0:
        return np.nan

    return float(np.mean(np.abs(frac_pos - mean_pred)))


def compute_robustness_score(
    model: Pipeline,
    X_test: pd.DataFrame,
    y_test: pd.Series
) -> float:
    X_noisy = X_test.copy()
    rng = np.random.default_rng(RANDOM_STATE)

    for col in X_noisy.columns:
        if pd.api.types.is_numeric_dtype(X_noisy[col]):
            std = X_noisy[col].std()
            noise = rng.normal(0.0, NOISE_STD * (std + 1e-9), size=len(X_noisy))
            X_noisy[col] = X_noisy[col] + noise

    clean_prob = get_probabilities(model, X_test)
    noisy_prob = get_probabilities(model, X_noisy)

    clean_auc = roc_auc_score(y_test, clean_prob)
    noisy_auc = roc_auc_score(y_test, noisy_prob)

    if clean_auc <= 1e-9:
        return 0.0

    return float(max(0.0, min(1.0, noisy_auc / clean_auc)))


def compute_group_metrics(
    y_true: pd.Series,
    y_prob: np.ndarray,
    group: pd.Series,
    threshold: float = CLASSIFICATION_THRESHOLD
) -> pd.DataFrame:
    df = pd.DataFrame({
        "y_true": np.asarray(y_true),
        "y_prob": np.asarray(y_prob),
        "group": np.asarray(group)
    })

    df["y_pred"] = (df["y_prob"] >= threshold).astype(int)

    rows = []

    for grp in sorted(df["group"].dropna().unique()):
        sub = df[df["group"] == grp].copy()

        if len(sub) < MIN_GROUP_SIZE:
            continue

        tn, fp, fn, tp = confusion_matrix(
            sub["y_true"],
            sub["y_pred"],
            labels=[0, 1]
        ).ravel()

        tpr = tp / (tp + fn + 1e-9)
        fpr = fp / (fp + tn + 1e-9)
        ppr = sub["y_pred"].mean()

        auc = np.nan
        if sub["y_true"].nunique() == 2:
            auc = roc_auc_score(sub["y_true"], sub["y_prob"])

        rows.append({
            "group": grp,
            "n": len(sub),
            "roc_auc": auc,
            "tpr": tpr,
            "fpr": fpr,
            "ppr": ppr
        })

    return pd.DataFrame(rows)


def compute_fairness_score(group_df: pd.DataFrame) -> Tuple[float, Dict[str, float]]:
    if group_df.shape[0] < 2:
        return 1.0, {
            "auc_gap": 0.0,
            "tpr_gap": 0.0,
            "fpr_gap": 0.0,
            "ppr_gap": 0.0
        }

    auc_gap = group_df["roc_auc"].max() - group_df["roc_auc"].min()
    tpr_gap = group_df["tpr"].max() - group_df["tpr"].min()
    fpr_gap = group_df["fpr"].max() - group_df["fpr"].min()
    ppr_gap = group_df["ppr"].max() - group_df["ppr"].min()

    mean_gap = np.nanmean([auc_gap, tpr_gap, fpr_gap, ppr_gap])
    fairness_score = max(0.0, 1.0 - mean_gap)

    return float(fairness_score), {
        "auc_gap": float(auc_gap),
        "tpr_gap": float(tpr_gap),
        "fpr_gap": float(fpr_gap),
        "ppr_gap": float(ppr_gap)
    }


def compute_trustworthiness_score(
    performance_score: float,
    robustness_score: float,
    fairness_score: float,
    calibration_score: float,
    interpretability_prior: float
) -> float:
    weights = compute_ahp_weights()

    score = (
        weights["performance"] * performance_score +
        weights["robustness"] * robustness_score +
        weights["fairness"] * fairness_score +
        weights["calibration"] * calibration_score +
        weights["interpretability"] * interpretability_prior
    )

    return round(float(score), 4)


def get_feature_importance(
    fitted_model: Pipeline,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    top_n: int = TOP_N_FEATURES
) -> pd.DataFrame:
    result = permutation_importance(
        fitted_model,
        X_test,
        y_test,
        n_repeats=10,
        random_state=RANDOM_STATE,
        scoring="roc_auc"
    )

    imp_df = pd.DataFrame({
        "feature": X_test.columns,
        "importance": result.importances_mean
    }).sort_values("importance", ascending=False)

    return imp_df.head(top_n).reset_index(drop=True)



AHP-derived weights:
performance: 0.459
robustness: 0.224
fairness: 0.169
calibration: 0.084
interpretability: 0.063


In [3]:
# ============================================================
# STYLED PLOTTING
# ============================================================

def save_conf_matrix(y_true, y_prob, title, filename, threshold=0.5):

    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(5,4.5))

    im = ax.imshow(cm, cmap="Blues")

    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")

    ax.set_xticks([0,1])
    ax.set_yticks([0,1])
    ax.set_xticklabels(["Negative","Positive"])
    ax.set_yticklabels(["Negative","Positive"])

    threshold = cm.max() / 2

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):

            color = "white" if cm[i,j] > threshold else "black"

            ax.text(
                j,
                i,
                f"{cm[i,j]}",
                ha="center",
                va="center",
                fontsize=12,
                fontweight="bold",
                color=color
            )

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Count")

    plt.tight_layout()
    plt.savefig(filename)
    plt.close()



def save_roc_curve(y_true, y_prob, title, filename):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, linewidth=2.2, label=f"ROC Curve (AUC = {auc:.3f})")
    plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1.5, label="Chance")

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title, pad=12, fontweight="bold")
    plt.legend(frameon=False, loc="lower right")
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()


def save_feature_importance(imp_df, title, filename):
    imp_df = imp_df.sort_values("importance", ascending=True)

    plt.figure(figsize=(7, 5))
    bars = plt.barh(imp_df["feature"], imp_df["importance"])

    plt.xlabel("Permutation Importance")
    plt.title(title, pad=12, fontweight="bold")

    for bar in bars:
        width = bar.get_width()
        plt.text(
            width + max(imp_df["importance"].max() * 0.02, 0.0005),
            bar.get_y() + bar.get_height() / 2,
            f"{width:.3f}",
            va="center",
            fontsize=9
        )

    plt.tight_layout()
    plt.savefig(filename)
    plt.close()


def save_calibration_plot(y_true, y_prob, title, filename):
    frac_pos, mean_pred = calibration_curve(
        y_true,
        y_prob,
        n_bins=10,
        strategy="uniform"
    )

    plt.figure(figsize=(6, 5))
    plt.plot(mean_pred, frac_pos, marker="o", linewidth=2, label="Model")
    plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1.5, label="Perfect calibration")

    plt.xlabel("Mean Predicted Probability")
    plt.ylabel("Observed Fraction of Positives")
    plt.title(title, pad=12, fontweight="bold")
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()


In [4]:
# ============================================================
# PDF REPORT EXPORT
# ============================================================

def export_pdf_report(
    output_pdf_path: str,
    final_results: pd.DataFrame,
    dataset_outputs: Dict[str, Dict[str, Dict[str, str]]],
    feature_tables: Dict[str, Dict[str, pd.DataFrame]],
    fairness_tables: Dict[str, Dict[str, pd.DataFrame]]
):
    with PdfPages(output_pdf_path) as pdf:
        # Cover page
        fig = plt.figure(figsize=(8.27, 11.69))
        fig.text(0.5, 0.92, "Trustworthy AI Evaluation Report",
                 ha="center", fontsize=20, fontweight="bold")
        fig.text(0.5, 0.88, "Healthcare Decision Support Models",
                 ha="center", fontsize=14)
        fig.text(0.5, 0.84, "Datasets: Heart Disease and Breast Cancer",
                 ha="center", fontsize=12)

        fig.text(
            0.1, 0.70,
            "This report summarizes model performance, robustness, fairness,\n"
            "calibration, interpretability, and composite trustworthiness.\n\n"
            "Models evaluated:\n"
            "- Logistic Regression\n"
            "- Random Forest\n"
            "- Gradient Boosting\n\n"
            "Outputs include:\n"
            "- Summary tables\n"
            "- Fairness/group metrics\n"
            "- Feature importance\n"
            "- ROC, confusion matrix, and calibration plots",
            fontsize=12
        )
        plt.axis("off")
        pdf.savefig(fig)
        plt.close()

        # Final results table
        fig, ax = plt.subplots(figsize=(14, 6))
        ax.axis("off")
        ax.set_title("Final Results Summary", fontsize=16, fontweight="bold", pad=15)

        show_df = final_results.copy().round(4)
        table = ax.table(
            cellText=show_df.values,
            colLabels=show_df.columns,
            loc="center"
        )
        table.auto_set_font_size(False)
        table.set_fontsize(8)
        table.scale(1, 1.4)

        pdf.savefig(fig)
        plt.close()

        # Dataset sections
        for dataset_name, models_dict in dataset_outputs.items():
            fig = plt.figure(figsize=(8.27, 11.69))
            fig.text(0.5, 0.93, dataset_name, ha="center", fontsize=18, fontweight="bold")
            fig.text(0.5, 0.89, "Model Evaluation Figures and Tables", ha="center", fontsize=12)
            plt.axis("off")
            pdf.savefig(fig)
            plt.close()

            for model_name, file_dict in models_dict.items():
                fig = plt.figure(figsize=(8.27, 11.69))
                gs = gridspec.GridSpec(2, 2, figure=fig)
                fig.suptitle(f"{dataset_name} - {model_name}", fontsize=16, fontweight="bold", y=0.98)

                img1 = plt.imread(file_dict["conf_matrix"])
                ax1 = fig.add_subplot(gs[0, 0])
                ax1.imshow(img1)
                ax1.axis("off")
                ax1.set_title("Confusion Matrix", fontsize=11)

                img2 = plt.imread(file_dict["roc_curve"])
                ax2 = fig.add_subplot(gs[0, 1])
                ax2.imshow(img2)
                ax2.axis("off")
                ax2.set_title("ROC Curve", fontsize=11)

                img3 = plt.imread(file_dict["feature_importance"])
                ax3 = fig.add_subplot(gs[1, 0])
                ax3.imshow(img3)
                ax3.axis("off")
                ax3.set_title("Feature Importance", fontsize=11)

                img4 = plt.imread(file_dict["calibration"])
                ax4 = fig.add_subplot(gs[1, 1])
                ax4.imshow(img4)
                ax4.axis("off")
                ax4.set_title("Calibration Plot", fontsize=11)

                plt.tight_layout(rect=[0, 0, 1, 0.96])
                pdf.savefig(fig)
                plt.close()

                if dataset_name in feature_tables and model_name in feature_tables[dataset_name]:
                    fig, ax = plt.subplots(figsize=(8.27, 11.69))
                    ax.axis("off")
                    ax.set_title(
                        f"{dataset_name} - {model_name} - Top Features",
                        fontsize=15,
                        fontweight="bold",
                        pad=15
                    )

                    df_feat = feature_tables[dataset_name][model_name].copy().round(4)
                    table = ax.table(
                        cellText=df_feat.values,
                        colLabels=df_feat.columns,
                        loc="center"
                    )
                    table.auto_set_font_size(False)
                    table.set_fontsize(10)
                    table.scale(1, 1.5)

                    pdf.savefig(fig)
                    plt.close()

                if dataset_name in fairness_tables and model_name in fairness_tables[dataset_name]:
                    fig, ax = plt.subplots(figsize=(8.27, 11.69))
                    ax.axis("off")
                    ax.set_title(
                        f"{dataset_name} - {model_name} - Fairness / Group Metrics",
                        fontsize=15,
                        fontweight="bold",
                        pad=15
                    )

                    df_fair = fairness_tables[dataset_name][model_name].copy().round(4)
                    table = ax.table(
                        cellText=df_fair.values,
                        colLabels=df_fair.columns,
                        loc="center"
                    )
                    table.auto_set_font_size(False)
                    table.set_fontsize(10)
                    table.scale(1, 1.5)

                    pdf.savefig(fig)
                    plt.close()

    print(f"Saved PDF report: {output_pdf_path}")



In [5]:
# ============================================================
# CORE EVALUATION
# ============================================================

def evaluate_dataset(
    dataset_name: str,
    X: pd.DataFrame,
    y: pd.Series,
    subgroup: pd.Series
):
    cv = RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=RANDOM_STATE
    )

    all_results = []
    importance_tables = {}
    fairness_tables = {}
    plot_files = {}

    print("\n" + "=" * 80)
    print(f"DATASET: {dataset_name}")
    print("=" * 80)

    for spec in MODEL_SPECS:
        fold_rows = []

        last_fitted_model = None
        last_X_test = None
        last_y_test = None
        last_prob = None
        last_group_test = None

        for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
            X_train = X.iloc[train_idx].copy()
            X_test = X.iloc[test_idx].copy()
            y_train = y.iloc[train_idx].copy()
            y_test = y.iloc[test_idx].copy()
            group_test = subgroup.iloc[test_idx].copy()

            model = make_pipeline(clone(spec.estimator))
            model.fit(X_train, y_train)

            y_prob = get_probabilities(model, X_test)

            perf = compute_performance(y_test, y_prob)
            auc_ci_low, auc_ci_high = bootstrap_auc_ci(y_test, y_prob)
            calibration_error = compute_calibration_error(y_test, y_prob)
            calibration_score = max(0.0, 1.0 - calibration_error) if not np.isnan(calibration_error) else 0.0
            robustness_score = compute_robustness_score(model, X_test, y_test)

            group_df = compute_group_metrics(y_test, y_prob, group_test)
            fairness_score, fairness_gaps = compute_fairness_score(group_df)

            trust_score = compute_trustworthiness_score(
                performance_score=perf["roc_auc"],
                robustness_score=robustness_score,
                fairness_score=fairness_score,
                calibration_score=calibration_score,
                interpretability_prior=spec.interpretability_prior
            )

            fold_rows.append({
                "dataset": dataset_name,
                "model": spec.name,
                "fold": fold_idx,
                "accuracy": perf["accuracy"],
                "precision": perf["precision"],
                "recall": perf["recall"],
                "f1": perf["f1"],
                "roc_auc": perf["roc_auc"],
                "auc_ci_low": auc_ci_low,
                "auc_ci_high": auc_ci_high,
                "brier_score": perf["brier_score"],
                "calibration_error": calibration_error,
                "robustness_score": robustness_score,
                "fairness_score": fairness_score,
                "auc_gap": fairness_gaps["auc_gap"],
                "tpr_gap": fairness_gaps["tpr_gap"],
                "fpr_gap": fairness_gaps["fpr_gap"],
                "ppr_gap": fairness_gaps["ppr_gap"],
                "interpretability_prior": spec.interpretability_prior,
                "trustworthiness_score": trust_score
            })

            last_fitted_model = model
            last_X_test = X_test
            last_y_test = y_test
            last_prob = y_prob
            last_group_test = group_test

        fold_df = pd.DataFrame(fold_rows)

        summary = (
            fold_df.drop(columns=["fold"])
            .groupby(["dataset", "model"], as_index=False)
            .mean(numeric_only=True)
        )

        all_results.append(summary)

        group_df = compute_group_metrics(last_y_test, last_prob, last_group_test)
        fairness_tables[spec.name] = group_df

        imp_df = get_feature_importance(last_fitted_model, last_X_test, last_y_test)
        importance_tables[spec.name] = imp_df

        base_name = f"{sanitize_name(dataset_name)}_{sanitize_name(spec.name)}"

        conf_path = os.path.join(OUTPUT_DIR, f"{base_name}_confusion_matrix.png")
        roc_path = os.path.join(OUTPUT_DIR, f"{base_name}_roc_curve.png")
        feat_path = os.path.join(OUTPUT_DIR, f"{base_name}_feature_importance.png")
        cal_path = os.path.join(OUTPUT_DIR, f"{base_name}_calibration.png")

        save_conf_matrix(
            last_y_test,
            last_prob,
            f"{dataset_name} - {spec.name} - Confusion Matrix",
            conf_path
        )
        save_roc_curve(
            last_y_test,
            last_prob,
            f"{dataset_name} - {spec.name} - ROC Curve",
            roc_path
        )
        save_feature_importance(
            imp_df,
            f"{dataset_name} - {spec.name} - Feature Importance",
            feat_path
        )
        save_calibration_plot(
            last_y_test,
            last_prob,
            f"{dataset_name} - {spec.name} - Calibration Plot",
            cal_path
        )

        plot_files[spec.name] = {
            "conf_matrix": conf_path,
            "roc_curve": roc_path,
            "feature_importance": feat_path,
            "calibration": cal_path
        }

        print(f"\nModel: {spec.name}")
        print(summary.round(4).to_string(index=False))

        if not group_df.empty:
            print("\nLast-fold group metrics:")
            print(group_df.round(4).to_string(index=False))

    result_df = pd.concat(all_results, ignore_index=True).sort_values(
        by=["dataset", "trustworthiness_score", "roc_auc"],
        ascending=[True, False, False]
    )

    return result_df, importance_tables, fairness_tables, plot_files



In [6]:
# ============================================================
# MAIN
# ============================================================

def main():
    heart_X, heart_y, heart_group = load_heart_dataset(HEART_PATH)
    breast_X, breast_y, breast_group = load_breast_dataset(BREAST_PATH)

    print("Heart dataset loaded:", heart_X.shape, heart_y.shape)
    print("Breast dataset loaded:", breast_X.shape, breast_y.shape)

    heart_results, heart_importances, heart_fairness, heart_plots = evaluate_dataset(
        dataset_name="Heart Disease",
        X=heart_X,
        y=heart_y,
        subgroup=heart_group
    )

    breast_results, breast_importances, breast_fairness, breast_plots = evaluate_dataset(
        dataset_name="Breast Cancer",
        X=breast_X,
        y=breast_y,
        subgroup=breast_group
    )

    final_results = pd.concat([heart_results, breast_results], ignore_index=True)

    final_csv = os.path.join(OUTPUT_DIR, "trustworthy_ai_results_improved.csv")
    final_results.to_csv(final_csv, index=False)

    for model_name, df in heart_importances.items():
        out_path = os.path.join(
            OUTPUT_DIR,
            f"heart_{sanitize_name(model_name)}_top_features.csv"
        )
        df.to_csv(out_path, index=False)

    for model_name, df in breast_importances.items():
        out_path = os.path.join(
            OUTPUT_DIR,
            f"breast_{sanitize_name(model_name)}_top_features.csv"
        )
        df.to_csv(out_path, index=False)

    for model_name, df in heart_fairness.items():
        out_path = os.path.join(
            OUTPUT_DIR,
            f"heart_{sanitize_name(model_name)}_fairness.csv"
        )
        df.to_csv(out_path, index=False)

    for model_name, df in breast_fairness.items():
        out_path = os.path.join(
            OUTPUT_DIR,
            f"breast_{sanitize_name(model_name)}_fairness.csv"
        )
        df.to_csv(out_path, index=False)

    feature_tables = {
        "Heart Disease": heart_importances,
        "Breast Cancer": breast_importances
    }

    fairness_tables = {
        "Heart Disease": heart_fairness,
        "Breast Cancer": breast_fairness
    }

    dataset_outputs = {
        "Heart Disease": heart_plots,
        "Breast Cancer": breast_plots
    }

    pdf_path = os.path.join(OUTPUT_DIR, "trustworthy_ai_evaluation_report.pdf")
    export_pdf_report(
        output_pdf_path=pdf_path,
        final_results=final_results,
        dataset_outputs=dataset_outputs,
        feature_tables=feature_tables,
        fairness_tables=fairness_tables
    )

    print("\n" + "=" * 80)
    print("FINAL RESULTS")
    print("=" * 80)
    print(final_results.round(4).to_string(index=False))

    print("\nSaved outputs in folder:", OUTPUT_DIR)
    print("Saved CSV:", final_csv)
    print("Saved PDF:", pdf_path)


if __name__ == "__main__":
    main()


Heart dataset loaded: (303, 13) (303,)
Breast dataset loaded: (569, 30) (569,)

DATASET: Heart Disease

Model: Logistic Regression
      dataset               model  accuracy  precision  recall     f1  roc_auc  auc_ci_low  auc_ci_high  brier_score  calibration_error  robustness_score  fairness_score  auc_gap  tpr_gap  fpr_gap  ppr_gap  interpretability_prior  trustworthiness_score
Heart Disease Logistic Regression    0.8375     0.8506  0.7914 0.8165   0.9059      0.8261       0.9712       0.1221              0.192            0.9954          0.8246   0.0605   0.1669    0.151   0.3234                     1.0                 0.9099

Last-fold group metrics:
 group  n  roc_auc    tpr    fpr    ppr
female 18   0.8036 0.5000 0.0000 0.1111
  male 42   0.8218 0.7917 0.1667 0.5238

Model: Random Forest
      dataset         model  accuracy  precision  recall     f1  roc_auc  auc_ci_low  auc_ci_high  brier_score  calibration_error  robustness_score  fairness_score  auc_gap  tpr_gap  fpr_gap  ppr

PermissionError: [Errno 13] Permission denied: 'trustworthy_ai_outputs\\trustworthy_ai_evaluation_report.pdf'